import libraries


In [2]:
import os
import random

# Path to the file
file_path = "backend/detection/model/real_news_paragraph.txt"

# Ensure the directory exists
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Check if file exists
if not os.path.exists(file_path):
    print(f"{file_path} not found. Generating file...")
    
    # Dynamically generate dataset
    subjects = ["The government", "A new study", "Scientists", "Experts", "The media"]
    actions = ["reveals", "confirms", "warns about", "predicts", "investigates"]
    objects = ["a major secret", "climate change effects", "a new virus", "political corruption", "economic collapse"]
    
    sentences = [f"{random.choice(subjects)} {random.choice(actions)} {random.choice(objects)}." for i in range(1000)]
    full_paragraph = "\n".join(sentences)
    
    # Write the dataset to file
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(full_paragraph)
    
    print(f"{file_path} generated successfully!")
else:
    print(f"{file_path} already exists. Using existing file.")

# Read the file
with open(file_path, "r", encoding="utf-8") as f:
    data = f.read()

print(f"Data loaded. Total sentences: {len(data.splitlines())}")


backend/detection/model/real_news_paragraph.txt not found. Generating file...
backend/detection/model/real_news_paragraph.txt generated successfully!
Data loaded. Total sentences: 1000


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import re,string,joblib
from nltk.corpus import stopwords

In [4]:
fake = pd.read_csv('../data/Fake.csv',)
true = pd.read_csv('../data/True.csv',)

In [5]:
fake['label']=0
true['label']=1

In [6]:
fake.head()

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0


In [7]:
true.head()

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1


merging

In [8]:
data = pd.concat([fake,true], axis = 0)
data["combined"] = (data["title"].fillna('') + " " + data["text"].fillna(''))
full_paragraph = " ".join(data["combined"].tolist())

In [9]:
with open("real_news_paragraph.txt", "w", encoding= "utf-8") as f:f.write(full_paragraph)

In [10]:
data.sample(10)

,title,text,subject,date,label,combined
13038,BREAKING…NYC: Stunning Video Captures Hillary ...,This IS NOT heat related. #HillarysHealth pic....,politics,"Sep 11, 2016",0,BREAKING…NYC: Stunning Video Captures Hillary ...
7337,Pollsters who predicted Trump win benefit from...,NEW YORK (Reuters) - A handful of small public...,politicsNews,"November 11, 2016",1,Pollsters who predicted Trump win benefit from...
20346,Number of Rohingya fleeing from Myanmar to Ban...,"COX S BAZAR, Bangladesh (Reuters) - An estimat...",worldnews,"September 12, 2017",1,Number of Rohingya fleeing from Myanmar to Ban...
4456,Chelsea Handler Hilariously Trashes Trump’s S...,The prospect of President Obama and his family...,News,"September 28, 2016",0,Chelsea Handler Hilariously Trashes Trump’s S...
3446,China Fires Back As Trump Drama ESCALATES Bef...,President-elect Donald Trump has yet to take o...,News,"December 12, 2016",0,China Fires Back As Trump Drama ESCALATES Bef...
1354,China says hopes Iran nuclear deal stays intac...,BEIJING (Reuters) - China said on Monday it ho...,politicsNews,"October 9, 2017",1,China says hopes Iran nuclear deal stays intac...
20334,Russian Islamic State fighter sentenced to han...,BAGHDAD (Reuters) - A Russian Islamic State fi...,worldnews,"September 12, 2017",1,Russian Islamic State fighter sentenced to han...
7124,Megyn Kelly Laughably Claims Fox News Does No...,You ll have a hard time not laughing out loud ...,News,"April 3, 2016",0,Megyn Kelly Laughably Claims Fox News Does No...
3941,BUSTED: Trump Supporter Used Poll Watcher Cre...,"Clearly, there is no low Trump supporters won ...",News,"November 4, 2016",0,BUSTED: Trump Supporter Used Poll Watcher Cre...
20295,NFL PLAYER POSTS Picture Of Cop’s Throat Being...,When Black Lives Matter supporters use the sam...,left-news,"Jul 11, 2016",0,NFL PLAYER POSTS Picture Of Cop’s Throat Being...


In [11]:
data = data.drop(["title","subject","date"], axis = 1)

In [12]:
print(data.columns)

Index(['text', 'label', 'combined'], dtype='object')


In [13]:
data.sample(5)

,text,label,combined
21249,This is a MUST watch from start to finish. The...,0,Dinesh D’Souza DESTROYS Leftist College Studen...
21272,Individual people can be bullied into submiss...,0,“WHITE STUDENT UNION” Groups Spring Up World-W...
678,LOS ANGELES (Reuters) - Convincing big U.S. d...,1,"As Trump targets immigrants, U.S. farm sector ..."
1534,Brave White House press secretary Sean Spicer ...,0,‘Turn The Lights Off!’: Spicer Literally Hid ...
18075,Unhinged Democrat protesters converged on the ...,0,ALT-LEFT ATTACKS PHOENIX POLICE…Karma Hits Bac...


In [14]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(r'\W+', ' ', text)
    return text.strip()


In [15]:
data["text"] = data["text"].apply(clean_text)

In [16]:
print(data.columns)



Index(['text', 'label', 'combined'], dtype='object')


In [17]:
x=data["combined"]
y=data["label"]

In [18]:
xtrain,xtest,ytrain,ytest = train_test_split(x,y,test_size=0.2,random_state=42)

In [19]:
vectorizer = TfidfVectorizer(max_df=0.7, stop_words='english')
xv_train = vectorizer.fit_transform(xtrain)
xv_test = vectorizer.transform(xtest)

In [20]:
model = LogisticRegression(max_iter=1000)
model.fit(xv_train, ytrain)

LogisticRegression(max_iter=1000)

In [21]:
whos

Variable                Type                  Data/Info
-------------------------------------------------------
LogisticRegression      type                  <class 'sklearn.linear_mo<...>stic.LogisticRegression'>
TfidfVectorizer         type                  <class 'sklearn.feature_e<...>on.text.TfidfVectorizer'>
actions                 list                  n=5
classification_report   function              <function classification_<...>rt at 0x000002242E5B19E0>
clean_text              function              <function clean_text at 0x00000224302137E0>
data                    DataFrame                                      <...>n[44914 rows x 3 columns]
f                       TextIOWrapper         <_io.TextIOWrapper name='<...>ode='w' encoding='utf-8'>
fake                    DataFrame                                      <...>n[23481 rows x 5 columns]
file_path               str                   backend/detection/model/real_news_paragraph.txt
full_paragraph          str               

In [22]:
prediction = model.predict(xv_test)
print("Accuracy:",model.score(xv_test,ytest))
print(model.score(xv_test,ytest))

Accuracy: 0.9869753979739508
0.9869753979739508


In [23]:
print(classification_report(ytest,prediction))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      4670
           1       0.98      0.99      0.99      4313

    accuracy                           0.99      8983
   macro avg       0.99      0.99      0.99      8983
weighted avg       0.99      0.99      0.99      8983



In [24]:
import joblib
joblib.dump(vectorizer, "vectorizer.jb")

['vectorizer.jb']

In [25]:
joblib.dump(model, "lr_model.jb")

['lr_model.jb']

In [26]:
sample = ["This is a test news article"]
sample_vec = vectorizer.transform(sample)
print(model.predict(sample_vec))

[0]


In [27]:
# Test with a real-like headline
sample_real = ["The government announced a new healthcare reform today."]
vec_real = vectorizer.transform(sample_real)
print("Prediction (1=real, 0=fake):", model.predict(vec_real))

# Test with a fake-like headline
sample_fake = ["Aliens landed in New York City last night and vanished!"]
vec_fake = vectorizer.transform(sample_fake)
print("Prediction (1=real, 0=fake):", model.predict(vec_fake))


Prediction (1=real, 0=fake): [1]
Prediction (1=real, 0=fake): [0]
